# Unificação e Pré-Processamento dos Dados de Estações – QualiAR

Este notebook tem como objetivo **carregar, unificar e preparar** as séries temporais diárias de variáveis meteorológicas e poluentes atmosféricos medidas nas **8 estações de monitoramento do DataRio** na cidade do Rio de Janeiro:

- BANGU  
- CAMPO_GRANDE  
- PEDRA_DE_GUARATIBA  
- IRAJA  
- SAO_CRISTOVAO  
- TIJUCA  
- CENTRO  
- COPACABANA  

## Propósito
1. **Leitura direta** dos arquivos tratados por dia a partir do repositório GitHub do projeto **QualiAR**.  
2. **Padronização** dos nomes de colunas, formatação de datas e inclusão de identificador da estação.  
3. **Concatenação** dos dados de todas as estações em um único `DataFrame`.  
4. (Etapas futuras) Possibilitar:
   - Análise exploratória conjunta da cidade.
   - Agregação diária para cálculo de médias/máximos por município.
   - Geração de estatísticas, gráficos e mapas para avaliação da qualidade do ar.

## Estrutura temporal
Os dados compreendem o período **2012 a 2024** e incluem:
- Variáveis meteorológicas: temperatura, umidade, precipitação (chuva), entre outras.
- Poluentes atmosféricos: CO, NO₂, NOx, SO₂, O₃, PM₁₀, PM₂.₅.
- Índice de Qualidade do Ar (AQI) e classificação qualitativa.

---


## Configurações e importações
Bibliotecas usadas e parâmetros gerais.

In [8]:
import pandas as pd
import glob
import os
import unicodedata
from pathlib import Path
import numpy as np

## Carregamento e Unificação das Estações

Nesta etapa:

1. Lemos os CSVs diários de cada estação diretamente do GitHub 
2. Garantimos que a coluna `data_dia` esteja no formato de data
3. Adicionamos o nome da estação quando não está no arquivo
4. Unimos todos os dados em um único DataFrame
5. Ordenamos por data e salvamos o resultado em `ESTACOES_UNIFICADAS_POR_DIA.csv`

In [2]:
def carregar_estacoes_github(estacoes):
    """
    Lê CSVs diretamente do repositório GitHub (modo raw) e concatena.
    
    estacoes: lista de nomes das estações (strings, ex.: ["BANGU", "CAMPO_GRANDE", ...])
    """
    base_url = "https://raw.githubusercontent.com/AILAB-CEFET-RJ/qualiar/main/data/DataRio/Estacoes_Tratadas_Por_Dia"
    
    dfs = []
    for est in estacoes:
        url = f"{base_url}/ESTACAO_{est}_POR_DIA.csv"
        print(f"Lendo: {url}")
        df = pd.read_csv(url, encoding="utf-8")

        if "data_dia" in df.columns:
            df["data_dia"] = pd.to_datetime(df["data_dia"], errors="coerce")
       
        if "nome_estacao" not in df.columns:
            df["nome_estacao"] = est
        
        dfs.append(df)

    df_all = pd.concat(dfs, ignore_index=True)
    return df_all

lista_estacoes = ["BANGU", "CAMPO_GRANDE", "PEDRA_DE_GUARATIBA", "IRAJA", "SAO_CRISTOVAO", "TIJUCA", "CENTRO", "COPACABANA"]

df_estacoes = carregar_estacoes_github(lista_estacoes)

df_estacoes.sort_values(by=["data_dia"], inplace=True)

Lendo: https://raw.githubusercontent.com/AILAB-CEFET-RJ/qualiar/main/data/DataRio/Estacoes_Tratadas_Por_Dia/ESTACAO_BANGU_POR_DIA.csv
Lendo: https://raw.githubusercontent.com/AILAB-CEFET-RJ/qualiar/main/data/DataRio/Estacoes_Tratadas_Por_Dia/ESTACAO_CAMPO_GRANDE_POR_DIA.csv
Lendo: https://raw.githubusercontent.com/AILAB-CEFET-RJ/qualiar/main/data/DataRio/Estacoes_Tratadas_Por_Dia/ESTACAO_PEDRA_DE_GUARATIBA_POR_DIA.csv
Lendo: https://raw.githubusercontent.com/AILAB-CEFET-RJ/qualiar/main/data/DataRio/Estacoes_Tratadas_Por_Dia/ESTACAO_IRAJA_POR_DIA.csv
Lendo: https://raw.githubusercontent.com/AILAB-CEFET-RJ/qualiar/main/data/DataRio/Estacoes_Tratadas_Por_Dia/ESTACAO_SAO_CRISTOVAO_POR_DIA.csv
Lendo: https://raw.githubusercontent.com/AILAB-CEFET-RJ/qualiar/main/data/DataRio/Estacoes_Tratadas_Por_Dia/ESTACAO_TIJUCA_POR_DIA.csv
Lendo: https://raw.githubusercontent.com/AILAB-CEFET-RJ/qualiar/main/data/DataRio/Estacoes_Tratadas_Por_Dia/ESTACAO_CENTRO_POR_DIA.csv
Lendo: https://raw.githubusercon

In [5]:
project_root = Path().resolve().parents[2]  

output_dir = project_root / "data" / "DataRio" / "Estacoes_Tratadas_Por_Dia"
output_dir.mkdir(parents=True, exist_ok=True)

output_csv_path = output_dir / f"ESTACOES_UNIFICADAS_POR_DIA.csv"
df_estacoes.to_csv(output_csv_path, index=False, encoding='utf-8')

print(f"Arquivo salvo em: {output_csv_path}")

Arquivo salvo em: C:\Users\jhter\OneDrive - cefet-rj.br\qualiar\data\DataRio\Estacoes_Tratadas_Por_Dia\ESTACOES_UNIFICADAS_POR_DIA.csv


## Agregação das medições para toda a cidade do Rio de Janeiro

Nesta etapa:

1. Carregamos o arquivo **`ESTACOES_UNIFICADAS_POR_DIA.csv`** contendo as medições diárias de todas as estações.
2. Removemos colunas que não são necessárias para a agregação (`nome_estacao`, `codnum`, `ano`, `mes`, `dia`, `lat`, `lon`, `Qualidade_do_Ar`).
3. Agrupamos os dados pela coluna `data_dia`.
4. Calculamos a **média** de todas as variáveis numéricas para representar os valores diários médios do município.
5. O resultado é um DataFrame (`df_cidade`) com uma linha por dia e colunas contendo as variáveis atmosféricas e poluentes.


In [ ]:
cols_to_drop = ["nome_estacao", "codnum", "ano", "mes", "dia", "lat", "lon", "Qualidade_do_Ar"]
df_estacoes = df_estacoes.drop(columns=[c for c in cols_to_drop if c in df_estacoes.columns])

df_estacoes["data_dia"] = pd.to_datetime(df_estacoes["data_dia"], errors="coerce")

df_cidade = df_estacoes.groupby("data_dia").mean(numeric_only=True).reset_index()

display(df_cidade.head())

,data_dia,chuva,temp,ur,co,no,no2,nox,so2,o3,pm10,pm2_5,AQI
0,2012-01-01,12.250,25.834571,92.165000,0.425833,3.613333,23.519667,27.129667,2.673333,23.059286,23.925143,14.619,17.0
1,2012-01-02,56.050,22.836286,95.588571,0.305333,12.675000,27.160333,39.842333,1.793833,20.136429,13.872000,5.083,13.0
2,2012-01-03,0.025,24.947875,76.139250,0.260143,17.175333,28.730667,45.882667,3.918500,15.718375,24.063625,4.208,19.0
3,2012-01-04,0.050,26.006250,72.904125,0.274571,24.745667,40.337667,65.049333,3.123667,25.002500,35.773375,15.729,34.0
4,2012-01-05,0.000,26.498125,75.514500,0.271286,16.643000,34.914000,51.558667,3.066000,33.646250,32.901000,10.917,37.0


In [11]:
for col in df_cidade.columns:
    if col not in ["data_dia", "AQI"]:
        df_cidade[col] = df_cidade[col].round(3)

if "AQI" in df_cidade.columns:
    df_cidade["AQI"] = df_cidade["AQI"].round(0).astype("Int64")

df_cidade["ano"] = df_cidade["data_dia"].dt.year
df_cidade["mes"] = df_cidade["data_dia"].dt.month
df_cidade["dia"] = df_cidade["data_dia"].dt.day

def qual_nivel(idx):
    if pd.isna(idx):
        return np.nan
    if 0 <= idx <= 40:
        return 1
    if 41 <= idx <= 80:
        return 2
    if 81 <= idx <= 120:
        return 3
    if 121 <= idx <= 200:
        return 4
    if 201 <= idx <= 400:
        return 5
    return np.nan

df_cidade["Qualidade_do_Ar"] = df_cidade["AQI"].apply(qual_nivel).astype("Int64")

cols = df_cidade.columns.tolist()

colunas_ordenadas = (
    ["data_dia", "ano", "mes", "dia"] +
    [c for c in cols if c not in ["data_dia", "ano", "mes", "dia"]]
)

df_cidade = df_cidade[colunas_ordenadas]

# Visualizar primeiras linhas
display(df_cidade.head())

,data_dia,ano,mes,dia,chuva,temp,ur,co,no,no2,nox,so2,o3,pm10,pm2_5,AQI,Qualidade_do_Ar
0,2012-01-01,2012,1,1,12.250,25.835,92.165,0.426,3.613,23.520,27.130,2.673,23.059,23.925,14.619,17,1
1,2012-01-02,2012,1,2,56.050,22.836,95.589,0.305,12.675,27.160,39.842,1.794,20.136,13.872,5.083,13,1
2,2012-01-03,2012,1,3,0.025,24.948,76.139,0.260,17.175,28.731,45.883,3.918,15.718,24.064,4.208,19,1
3,2012-01-04,2012,1,4,0.050,26.006,72.904,0.275,24.746,40.338,65.049,3.124,25.002,35.773,15.729,34,1
4,2012-01-05,2012,1,5,0.000,26.498,75.514,0.271,16.643,34.914,51.559,3.066,33.646,32.901,10.917,37,1


In [12]:
project_root = Path().resolve().parents[2]  

output_dir = project_root / "data" / "DataRio" 
output_dir.mkdir(parents=True, exist_ok=True)

output_csv_path = output_dir / f"QUALIAR_RIO_DE_JANEIRO.csv"
df_cidade.to_csv(output_csv_path, index=False, encoding='utf-8')

print(f"Arquivo salvo em: {output_csv_path}")

Arquivo salvo em: C:\Users\jhter\OneDrive - cefet-rj.br\qualiar\data\DataRio\QUALIAR_RIO_DE_JANEIRO.csv
